# Notebook 04: Model Training and Statistical Comparison

Sprints 3-4 — Train baseline and alternative models across rolling-origin CV folds, perform statistical comparison.

In [1]:
import sys
import logging
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# Parent of notebooks/ is the project root
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import load_config
from src.data.data_splitter import RollingOriginSplitter, filter_active_products
from src.evaluation.metrics import evaluate_model, evaluate_across_folds
from src.models.baselines import SeasonalNaiveForecaster, RidgeForecaster, SARIMAForecaster

logging.basicConfig(level=logging.INFO)
config = load_config()
SEED = config["random_seed"]
np.random.seed(SEED)
print(f"Config loaded. Random seed: {SEED}")
print(f"Project root: {PROJECT_ROOT}")

Config loaded. Random seed: 42
Project root: /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final


## 1. Load Feature Data

In [2]:
pdf = pd.read_parquet(PROJECT_ROOT / "data" / "features")
pdf["date"] = pd.to_datetime(pdf["date"])
pdf = pdf.dropna().reset_index(drop=True)
print(f"Shape: {pdf.shape}")
print(f"Date range: {pdf['date'].min()} to {pdf['date'].max()}")
n_products = pdf.groupby(["subcategory", "sizing", "tactical"]).ngroups
print(f"Products: {n_products}")

Shape: (3552, 54)
Date range: 2020-01-01 00:00:00 to 2023-12-01 00:00:00
Products: 74


## 1.5 Product Segmentation

Products with fewer than 12 months of non-zero demand are excluded — insufficient data for seasonal time-series modeling. This removes 19 of 74 products (6 with zero total sales, 13 sporadic) while retaining >99.9% of total volume.

In [3]:
filter_config = config["data"]["product_filter"]
pdf_active, product_summary = filter_active_products(
    pdf,
    min_nonzero_months=filter_config["min_nonzero_months"],
    target_column=filter_config["target_column"],
    product_key=filter_config["product_key"],
)

n_active = product_summary["is_active"].sum()
n_inactive = len(product_summary) - n_active
vol_pct = pdf_active["quantity"].sum() / pdf["quantity"].sum() * 100

print(f"Active products: {n_active} / {len(product_summary)}")
print(f"Inactive products dropped: {n_inactive}")
print(f"Volume retained: {vol_pct:.1f}%")
print(f"Rows: {len(pdf)} → {len(pdf_active)}")

# Show inactive products
inactive = product_summary[~product_summary["is_active"]].sort_values("total_qty", ascending=False)
print(f"\nDropped products ({len(inactive)}):")
inactive

INFO:src.data.data_splitter:Product filter: 55/74 active (threshold=12 non-zero months), 100.0% volume retained (3552 → 2640 rows)


Active products: 55 / 74
Inactive products dropped: 19
Volume retained: 100.0%
Rows: 3552 → 2640

Dropped products (19):


,subcategory,sizing,tactical,total_qty,nonzero_months,total_months,is_active
45,Semiautomatic,223 REM/5.56 NATO,Y,928,11,48,False
31,Revolver,410 GA,N,348,10,48,False
71,Single Shot,410 GA,NA,127,11,48,False
32,Rifle/Shotgun,20 GA,N,26,5,48,False
47,Semiautomatic,28 GA,NA,24,4,48,False
72,Single Shot,410 GA,Y,19,7,48,False
16,Over/Under,45/410 GA,N,10,2,48,False
64,Single Shot,12 GA,NA,6,3,48,False
28,Pump Action,410 GA,NA,5,1,48,False
30,Pump Action,OTHER,Y,3,3,48,False


## 2. Rolling-Origin CV Setup

With 48 months of data (Jan 2020 - Dec 2023), min_train=24, horizon=3, step=3:
(48 - 24 - 3) / 3 + 1 = **8 folds** (not ~12 as originally estimated, due to the 12-month warm-up drop).

In [4]:
cv_config = config["evaluation"]["cv"]
splitter = RollingOriginSplitter(
    min_train_months=cv_config["min_train_months"],
    horizon_months=cv_config["horizon_months"],
    step_months=cv_config["step_months"],
)
folds = splitter.split(pdf_active)
fold_info = splitter.get_fold_info(pdf_active)
print(f"Number of folds: {len(folds)}")
pd.DataFrame([vars(fi) for fi in fold_info])

INFO:src.data.data_splitter:Splitting 2640 rows across 48 unique dates (min_train=24, horizon=3, step=3)


INFO:src.data.data_splitter:Created 8 folds


INFO:src.data.data_splitter:Generated fold info for 8 folds


Number of folds: 8


,fold_id,train_start,train_end,val_start,val_end,train_size,val_size
0,1,2020-01-01,2021-12-01,2022-01-01,2022-03-01,1320,165
1,2,2020-01-01,2022-03-01,2022-04-01,2022-06-01,1485,165
2,3,2020-01-01,2022-06-01,2022-07-01,2022-09-01,1650,165
3,4,2020-01-01,2022-09-01,2022-10-01,2022-12-01,1815,165
4,5,2020-01-01,2022-12-01,2023-01-01,2023-03-01,1980,165
5,6,2020-01-01,2023-03-01,2023-04-01,2023-06-01,2145,165
6,7,2020-01-01,2023-06-01,2023-07-01,2023-09-01,2310,165
7,8,2020-01-01,2023-09-01,2023-10-01,2023-12-01,2475,165


## 3. Define Feature Columns

In [5]:
NON_FEATURE_COLS = {
    "date", "subcategory", "sizing", "tactical", "sizing_grouped",
    "quantity", "amount", "transaction_count", "subcat_total_qty", "subcat_share",
}
# Rolling and change features include the current period's quantity in their
# computation (window includes row 0, change = quantity - lag), so they leak
# the target into supervised models. Exclude them from Ridge/XGBoost features.
LEAKING_COLS = {
    "quantity_yoy_change", "quantity_mom_change",
    "quantity_ma_3", "quantity_std_3",
    "quantity_ma_6", "quantity_std_6",
    "quantity_ma_12", "quantity_std_12",
}
EXCLUDE_COLS = NON_FEATURE_COLS | LEAKING_COLS
feature_cols = [c for c in pdf_active.columns if c not in EXCLUDE_COLS and pdf_active[c].dtype in ("float64", "int64", "int32", "float32")]
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
TARGET = "quantity"

Feature columns (36): ['barrel_length', 'avg_price', 'month', 'quarter', 'year', 'month_sin', 'month_cos', 'covid_period', 'post_covid', 'is_march', 'off_season', 'months_to_hunting', 'is_waterfowl_season', 'is_upland_season', 'is_turkey_spring_season', 'is_turkey_fall_season', 'is_dove_season', 'is_deer_season', 'hunting_intensity', 'quantity_lag_1', 'quantity_lag_3', 'quantity_lag_6', 'quantity_lag_12', 'avg_price_lag_1', 'is_tactical', 'is_tactical_na', 'is_semiautomatic', 'is_single_shot', 'is_over_under', 'is_lever_action', 'is_side_by_side', 'is_bolt_action', 'is_rifle_shotgun', 'is_revolver', 'is_12ga', 'time_index']


## 4. Baseline 1: Seasonal Naive

In [6]:
sn_fold_metrics = []
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_df = pdf_active.iloc[train_idx]
    val_df = pdf_active.iloc[val_idx]
    all_preds, all_actuals = [], []
    for (subcat, sizing, tact), group in val_df.groupby(["subcategory", "sizing", "tactical"]):
        train_series = train_df.query(
            "subcategory == @subcat and sizing == @sizing and tactical == @tact"
        )[TARGET].values
        model = SeasonalNaiveForecaster(season_length=12)
        model.fit(train_series)
        preds = model.predict(horizon=len(group))
        all_preds.extend(preds)
        all_actuals.extend(group[TARGET].values)
    metrics = evaluate_model(np.array(all_actuals), np.array(all_preds))
    metrics["fold"] = fold_idx + 1
    sn_fold_metrics.append(metrics)
    print(f"Fold {fold_idx+1}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, MAPE={metrics['mape']:.2f}%")

sn_summary = evaluate_across_folds([{k: v for k, v in m.items() if k != "fold"} for m in sn_fold_metrics])
print("\nSeasonal Naive Summary:")
sn_summary

INFO:src.models.baselines:SeasonalNaiveForecaster fit on 24 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 24 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 24 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 24 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 24 observations


...

INFO:src.models.baselines:SeasonalNaiveForecaster fit on 45 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 45 observations


INFO:src.models.baselines:SeasonalNaiveForecaster fit on 45 observations


Fold 8: RMSE=1417.53, MAE=420.05, MAPE=219.56%

Seasonal Naive Summary:


,mean,std,min,max
rmse,1735.594363,1307.107877,561.565744,4143.502984
mae,534.171212,320.550147,223.624242,1085.993939
mape,907.338860,1083.391667,86.994510,2857.907587


## 5. Baseline 2: Ridge Regression

In [7]:
ridge_fold_metrics = []
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_df = pdf_active.iloc[train_idx]
    val_df = pdf_active.iloc[val_idx]
    X_train = train_df[feature_cols].values
    y_train = train_df[TARGET].values
    X_val = val_df[feature_cols].values
    y_val = val_df[TARGET].values
    ridge_alpha = config["models"]["baselines"]["ridge"]["alpha"]
    model = RidgeForecaster(alpha=ridge_alpha, random_state=SEED)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    metrics = evaluate_model(y_val, preds)
    metrics["fold"] = fold_idx + 1
    ridge_fold_metrics.append(metrics)
    print(f"Fold {fold_idx+1}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, MAPE={metrics['mape']:.2f}%")

ridge_summary = evaluate_across_folds([{k: v for k, v in m.items() if k != "fold"} for m in ridge_fold_metrics])
print("\nRidge Regression Summary:")
ridge_summary

INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 1320 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 1485 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 1650 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 1815 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 1980 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 2145 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 2310 samples


INFO:src.models.baselines:RidgeForecaster fit with alpha=1.0000 on 2475 samples


Fold 1: RMSE=1090.80, MAE=551.03, MAPE=1424.23%
Fold 2: RMSE=707.27, MAE=460.29, MAPE=2772.04%
Fold 3: RMSE=540.22, MAE=294.58, MAPE=705.57%
Fold 4: RMSE=544.83, MAE=249.27, MAPE=688.05%
Fold 5: RMSE=520.28, MAE=288.25, MAPE=1389.25%
Fold 6: RMSE=479.27, MAE=333.24, MAPE=1624.20%
Fold 7: RMSE=348.57, MAE=200.11, MAPE=891.04%
Fold 8: RMSE=1665.67, MAE=483.41, MAPE=549.99%

Ridge Regression Summary:


,mean,std,min,max
rmse,737.115672,435.629334,348.571631,1665.672716
mae,357.522125,125.185039,200.114415,551.030040
mape,1255.545718,730.672550,549.988607,2772.035607


## 6. Baseline 3: SARIMA

SARIMA is fit per-product (univariate). Convergence failures are caught and fall back to training mean.

In [8]:
sarima_fold_metrics = []
total_failures = 0
sarima_order = tuple(config["models"]["baselines"]["sarima"]["order"])
sarima_seasonal = tuple(config["models"]["baselines"]["sarima"]["seasonal_order"])

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_df = pdf_active.iloc[train_idx]
    val_df = pdf_active.iloc[val_idx]
    all_preds, all_actuals = [], []
    fold_failures = 0
    for (subcat, sizing, tact), group in val_df.groupby(["subcategory", "sizing", "tactical"]):
        train_series = train_df.query(
            "subcategory == @subcat and sizing == @sizing and tactical == @tact"
        )[TARGET].values
        try:
            model = SARIMAForecaster(order=sarima_order, seasonal_order=sarima_seasonal)
            model.fit(train_series)
            preds = model.predict(horizon=len(group))
        except Exception:
            preds = np.full(len(group), np.mean(train_series))
            fold_failures += 1
        all_preds.extend(preds)
        all_actuals.extend(group[TARGET].values)
    total_failures += fold_failures
    metrics = evaluate_model(np.array(all_actuals), np.array(all_preds))
    metrics["fold"] = fold_idx + 1
    sarima_fold_metrics.append(metrics)
    print(f"Fold {fold_idx+1}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, MAPE={metrics['mape']:.2f}% (failures: {fold_failures})")

print(f"\nTotal convergence failures: {total_failures}")
sarima_summary = evaluate_across_folds([{k: v for k, v in m.items() if k != "fold"} for m in sarima_fold_metrics])
print("\nSARIMA Summary:")
sarima_summary

INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


...

INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


INFO:src.models.baselines:SARIMAForecaster fit with order=(1, 1, 1) seasonal_order=(1, 1, 1, 12)


Fold 8: RMSE=1480.69, MAE=519.57, MAPE=344.69% (failures: 0)

Total convergence failures: 0

SARIMA Summary:


,mean,std,min,max
rmse,283160.213533,587570.411052,430.847551,1.654647e+06
mae,23288.396877,46895.434994,205.732734,1.321673e+05
mape,11190.897272,21930.178803,344.688182,6.471811e+04


## 7. Summary Comparison

In [9]:
comparison = pd.DataFrame({
    "Seasonal Naive": sn_summary["mean"],
    "Ridge": ridge_summary["mean"],
    "SARIMA": sarima_summary["mean"],
})
comparison.index.name = "Metric"
print("Mean Metrics Across 8 Folds:")
comparison

Mean Metrics Across 8 Folds:


,Seasonal Naive,Ridge,SARIMA
Metric,,,
rmse,1735.594363,737.115672,283160.213533
mae,534.171212,357.522125,23288.396877
mape,907.338860,1255.545718,11190.897272


## 8. Save Results

In [10]:
results = {
    "seasonal_naive": sn_fold_metrics,
    "ridge": ridge_fold_metrics,
    "sarima": sarima_fold_metrics,
}
output_dir = PROJECT_ROOT / "models" / "trained"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "baseline_fold_metrics.pkl"
with open(output_path, "wb") as f:
    pickle.dump(results, f)
print(f"Saved per-fold metrics to {output_path}")

Saved per-fold metrics to /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/models/trained/baseline_fold_metrics.pkl


## 9. XGBoost Training

Sprint 4 — Tune XGBoost with Optuna, then evaluate across all CV folds.

In [11]:
from src.models.xgboost_model import XGBoostForecaster, XGBoostConfig, tune_xgboost
from src.models.lightgbm_model import LightGBMForecaster, LightGBMConfig, tune_lightgbm
from src.evaluation.statistical_tests import paired_ttest, format_hypothesis_report
from src.data.data_splitter import temporal_train_test_split

In [12]:
# Optuna tuning using temporal train/val split
train_df_tune, val_df_tune, _ = temporal_train_test_split(pdf_active)
X_train_tune = train_df_tune[feature_cols]
y_train_tune = train_df_tune[TARGET]
X_val_tune = val_df_tune[feature_cols]
y_val_tune = val_df_tune[TARGET]

n_trials = config["optuna"]["n_trials"]
print(f"Running XGBoost Optuna tuning ({n_trials} trials)...")
xgb_best_config, xgb_best_metrics, xgb_best_model = tune_xgboost(
    X_train_tune, y_train_tune, X_val_tune, y_val_tune,
    n_trials=n_trials, random_state=SEED,
)
print(f"Best XGBoost config: {xgb_best_config}")
print(f"Best validation MAPE: {xgb_best_metrics['mape']:.2f}%")

INFO:src.data.data_splitter:Temporal split: train=1980 rows (to 2022-12-31), val=330 rows (to 2023-06-30), test=330 rows (after 2023-06-30)


/Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running XGBoost Optuna tuning (50 trials)...


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1980 samples


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1980 samples


...

INFO:src.models.xgboost_model:XGBoostForecaster fit on 1980 samples


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1980 samples


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1980 samples


INFO:src.models.xgboost_model:XGBoost tuning complete: best MAPE=118.79%


Best XGBoost config: XGBoostConfig(n_estimators=500, max_depth=7, learning_rate=0.08112100900239375, subsample=0.7586222102197387, colsample_bytree=0.9430100301869281, min_child_weight=1, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0, random_state=42, n_jobs=-1)
Best validation MAPE: 118.79%


In [13]:
# Run XGBoost across all 8 CV folds with best config
xgb_fold_metrics = []
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_df = pdf_active.iloc[train_idx]
    val_df = pdf_active.iloc[val_idx]
    X_train = train_df[feature_cols]
    y_train = train_df[TARGET]
    X_val = val_df[feature_cols]
    y_val = val_df[TARGET]

    model = XGBoostForecaster(xgb_best_config)
    model.fit(X_train, y_train, X_val, y_val)
    preds = model.predict(X_val)
    metrics = evaluate_model(y_val.values, preds)
    metrics["fold"] = fold_idx + 1
    xgb_fold_metrics.append(metrics)
    print(f"Fold {fold_idx+1}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, MAPE={metrics['mape']:.2f}%")

xgb_summary = evaluate_across_folds([{k: v for k, v in m.items() if k != "fold"} for m in xgb_fold_metrics])
print("\nXGBoost Summary:")
xgb_summary

INFO:src.models.xgboost_model:XGBoostForecaster fit on 1320 samples


Fold 1: RMSE=751.98, MAE=476.09, MAPE=1127.53%


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1485 samples


Fold 2: RMSE=658.60, MAE=409.15, MAPE=2930.10%


INFO:src.models.xgboost_model:XGBoostForecaster fit on 1650 samples


...

INFO:src.models.xgboost_model:XGBoostForecaster fit on 2310 samples


Fold 6: RMSE=405.01, MAE=170.88, MAPE=161.58%
Fold 7: RMSE=327.72, MAE=163.77, MAPE=864.68%


INFO:src.models.xgboost_model:XGBoostForecaster fit on 2475 samples


Fold 8: RMSE=1444.54, MAE=400.98, MAPE=532.63%

XGBoost Summary:


,mean,std,min,max
rmse,646.228094,350.749699,327.715400,1444.538268
mae,289.595378,124.514680,163.769519,476.087456
mape,938.334890,900.460924,135.488876,2930.101606


### XGBoost Analysis

XGBoost achieves the lowest mean RMSE (646.23) and MAE (289.60) across all 8 folds, outperforming both baselines on absolute error metrics. MAPE remains high (938.33%) due to low-volume SKUs — see Discussion in Notebook 05.

## 10. LightGBM Training

Tune LightGBM with Optuna, then evaluate across all CV folds.

In [14]:
print(f"Running LightGBM Optuna tuning ({n_trials} trials)...")
lgb_best_config, lgb_best_metrics, lgb_best_model = tune_lightgbm(
    X_train_tune, y_train_tune, X_val_tune, y_val_tune,
    n_trials=n_trials, random_state=SEED,
)
print(f"Best LightGBM config: {lgb_best_config}")
print(f"Best validation MAPE: {lgb_best_metrics['mape']:.2f}%")

Running LightGBM Optuna tuning (50 trials)...
Training until validation scores don't improve for 50 rounds


INFO:src.models.lightgbm_model:LightGBMForecaster fit on 1980 samples


INFO:src.models.lightgbm_model:LightGBMForecaster fit on 1980 samples


Early stopping, best iteration is:
[41]	valid_0's l2: 228643
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's l2: 223905
Training until validation scores don't improve for 50 rounds


INFO:src.models.lightgbm_model:LightGBMForecaster fit on 1980 samples


...

INFO:src.models.lightgbm_model:LightGBMForecaster fit on 1980 samples


Early stopping, best iteration is:
[53]	valid_0's l2: 230097
Training until validation scores don't improve for 50 rounds


INFO:src.models.lightgbm_model:LightGBMForecaster fit on 1980 samples


INFO:src.models.lightgbm_model:LightGBM tuning complete: best MAPE=135.41%


Early stopping, best iteration is:
[100]	valid_0's l2: 230601
Best LightGBM config: LightGBMConfig(n_estimators=600, max_depth=7, learning_rate=0.08480284744011878, num_leaves=45, subsample=0.9223936245353436, colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.0, reg_lambda=0.0, random_state=42, n_jobs=-1)
Best validation MAPE: 135.41%


In [ ]:
import io, contextlib

# Run LightGBM across all 8 CV folds with best config
lgb_fold_metrics = []
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_df = pdf_active.iloc[train_idx]
    val_df = pdf_active.iloc[val_idx]
    X_train = train_df[feature_cols]
    y_train = train_df[TARGET]
    X_val = val_df[feature_cols]
    y_val = val_df[TARGET]

    model = LightGBMForecaster(lgb_best_config)
    with contextlib.redirect_stdout(io.StringIO()):
        model.fit(X_train, y_train, X_val, y_val)
    preds = model.predict(X_val)
    metrics = evaluate_model(y_val.values, preds)
    metrics["fold"] = fold_idx + 1
    lgb_fold_metrics.append(metrics)
    print(f"Fold {fold_idx+1}: RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, MAPE={metrics['mape']:.2f}%")

lgb_summary = evaluate_across_folds([{k: v for k, v in m.items() if k != "fold"} for m in lgb_fold_metrics])
print("\nLightGBM Summary:")
lgb_summary

### LightGBM Analysis

LightGBM performs comparably to XGBoost (RMSE: 672.95, MAE: 309.10) but with slightly higher variance across folds. Both gradient boosting models substantially improve over Ridge on RMSE/MAE while showing similar MAPE due to sparse demand inflation.

## 11. Feature Importance

Top features from the best XGBoost model.

In [16]:
importance_df = xgb_best_model.get_feature_importance()
print("Top 15 Features (XGBoost):")
importance_df.head(15)

Top 15 Features (XGBoost):


,feature,importance
0,quantity_lag_1,0.644397
1,quantity_lag_3,0.147672
2,covid_period,0.020987
3,time_index,0.020621
4,year,0.017351
5,month_cos,0.013963
6,is_tactical,0.013270
7,quantity_lag_6,0.012612
8,avg_price,0.011086
9,month_sin,0.009566


## 12. Statistical Testing

Paired t-tests comparing model MAPE distributions across folds.

In [17]:
# Extract per-fold MAPE arrays
sn_mapes = np.array([m["mape"] for m in sn_fold_metrics])
ridge_mapes = np.array([m["mape"] for m in ridge_fold_metrics])
xgb_mapes = np.array([m["mape"] for m in xgb_fold_metrics])
lgb_mapes = np.array([m["mape"] for m in lgb_fold_metrics])

alpha = config["evaluation"]["statistical_test"]["alpha"]

# Test 1: XGBoost vs Ridge
xgb_vs_ridge = paired_ttest(ridge_mapes, xgb_mapes, alpha=alpha)
print(format_hypothesis_report(xgb_vs_ridge, "Ridge", "XGBoost", "MAPE"))
print("=" * 60)

# Test 2: LightGBM vs Ridge
lgb_vs_ridge = paired_ttest(ridge_mapes, lgb_mapes, alpha=alpha)
print(format_hypothesis_report(lgb_vs_ridge, "Ridge", "LightGBM", "MAPE"))
print("=" * 60)

# Test 3: XGBoost vs Seasonal Naive
xgb_vs_sn = paired_ttest(sn_mapes, xgb_mapes, alpha=alpha)
print(format_hypothesis_report(xgb_vs_sn, "Seasonal Naive", "XGBoost", "MAPE"))

Paired t-test: XGBoost vs Ridge (MAPE)
H0: mean(XGBoost) >= mean(Ridge)
HA: mean(XGBoost) <  mean(Ridge)

               Ridge: 1255.5457 +/- 683.4816
             XGBoost: 938.3349 +/- 842.3041

t-statistic: -1.2978
p-value:     0.117735
95% CI:      [-inf, 145.8525]

Conclusion: Fail to reject H0 at alpha=0.05: no significant difference between XGBoost and Ridge.
Paired t-test: LightGBM vs Ridge (MAPE)
H0: mean(LightGBM) >= mean(Ridge)
HA: mean(LightGBM) <  mean(Ridge)

               Ridge: 1255.5457 +/- 683.4816
            LightGBM: 1062.2992 +/- 911.7665

t-statistic: -0.7968
p-value:     0.225881
95% CI:      [-inf, 266.2637]

Conclusion: Fail to reject H0 at alpha=0.05: no significant difference between LightGBM and Ridge.
Paired t-test: XGBoost vs Seasonal Naive (MAPE)
H0: mean(XGBoost) >= mean(Seasonal Naive)
HA: mean(XGBoost) <  mean(Seasonal Naive)

      Seasonal Naive: 907.3389 +/- 1013.4201
             XGBoost: 938.3349 +/- 842.3041

t-statistic: 0.0870
p-value:     0.5

## 13. Updated Summary Comparison

Mean metrics across 8 folds for all 5 models.

In [18]:
comparison = pd.DataFrame({
    "Seasonal Naive": sn_summary["mean"],
    "Ridge": ridge_summary["mean"],
    "SARIMA": sarima_summary["mean"],
    "XGBoost": xgb_summary["mean"],
    "LightGBM": lgb_summary["mean"],
})
comparison.index.name = "Metric"
print("Mean Metrics Across 8 Folds (All 5 Models):")
comparison

Mean Metrics Across 8 Folds (All 5 Models):


,Seasonal Naive,Ridge,SARIMA,XGBoost,LightGBM
Metric,,,,,
rmse,1735.594363,737.115672,283160.213533,646.228094,672.952600
mae,534.171212,357.522125,23288.396877,289.595378,309.098339
mape,907.338860,1255.545718,11190.897272,938.334890,1062.299159


## 14. Save All Results

Persist all fold metrics, best configs, and t-test results.

In [19]:
import yaml

all_results = {
    "seasonal_naive": sn_fold_metrics,
    "ridge": ridge_fold_metrics,
    "sarima": sarima_fold_metrics,
    "xgboost": xgb_fold_metrics,
    "lightgbm": lgb_fold_metrics,
}
output_path = PROJECT_ROOT / "models" / "trained" / "all_fold_metrics.pkl"
with open(output_path, "wb") as f:
    pickle.dump(all_results, f)
print(f"Saved all fold metrics to {output_path}")

# Save best configs
config_dir = PROJECT_ROOT / "models" / "configs"
config_dir.mkdir(parents=True, exist_ok=True)
from dataclasses import asdict
with open(config_dir / "xgboost_best_config.yaml", "w") as f:
    yaml.dump(asdict(xgb_best_config), f)
with open(config_dir / "lightgbm_best_config.yaml", "w") as f:
    yaml.dump(asdict(lgb_best_config), f)
print(f"Saved best configs to {config_dir}")

# Save t-test results
ttest_results = {
    "xgboost_vs_ridge": xgb_vs_ridge,
    "lightgbm_vs_ridge": lgb_vs_ridge,
    "xgboost_vs_seasonal_naive": xgb_vs_sn,
}
with open(PROJECT_ROOT / "models" / "trained" / "ttest_results.pkl", "wb") as f:
    pickle.dump(ttest_results, f)
print("Saved t-test results")

Saved all fold metrics to /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/models/trained/all_fold_metrics.pkl
Saved best configs to /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/models/configs
Saved t-test results
